In [31]:
from neuralbench.registry import ALL_MODELS, TASKS
print("EEG tasks:", ", ".join(TASKS["eeg"]))
print("Available models:", ", ".join(ALL_MODELS))

classic_models = [
    "shallow_fbcsp_net",
    "simpleconv_time_agg",
    "eegnet",
    "deep4net",
    "eegconformer",
    "atcnet",
    "bdtcn",
    "ctnet",
]

foundation_models = [
    "bendr",
    "biot",
    "cbramod",
    "labram",
    "luna",
    "reve",
]

print("Classic models:")
for model in classic_models:
    print("  ", model)

print("\nFoundation models:")
for model in foundation_models:
    print("  ", model)

EEG tasks: age, artifact, audiovisual_stimulus, clinical_event, cvep, dementia_diagnosis, depression_diagnosis, emotion, ern, image, lrp, mental_arithmetic, mental_imagery, mental_workload, mismatch_negativity, motor_execution, motor_imagery, n170, n2pc, n400, p3, parkinsons_diagnosis, pathology, psychopathology, reaction_time, schizophrenia_diagnosis, seizure, sentence, sex, sleep_arousal, sleep_onset, sleep_stage, speech, ssvep, typing, video, word
Available models: atcnet, bdtcn, bendr, biot, cbramod, chance, cospectra_log_lr, cov_ts_lr, cov_ts_ridge, ctnet, deep4net, dummy, eegconformer, eegnet, emg2qwerty, fmri_linear, fmri_mlp, labram, luna, mae, neuropose, reve, sensingdynamics, shallow_fbcsp_net, simpleconv, simpleconv_time_agg, vemg2pose, xdawn_ts_lr
Classic models:
   shallow_fbcsp_net
   simpleconv_time_agg
   eegnet
   deep4net
   eegconformer
   atcnet
   bdtcn
   ctnet

Foundation models:
   bendr
   biot
   cbramod
   labram
   luna
   reve


In [ ]:
from neuralbench import run_benchmark
results = run_benchmark(device = 'eeg',
                        task = 'audiovisual_stimulus', 
                        model = ['deep4net', 'eegnet'])

In [19]:
results = run_benchmark(
    device="eeg",
    task="audiovisual_stimulus",
    model = ["eegnet", "deep4net"],
    plot_cached=True,
)

INFO:neuralbench.cli:--- PREPARING GLOBAL PLOTS AND TABLES ---
INFO:neuralbench.aggregator:Saved computational stats to D:\Foundation Challenge 2026\results\outputs\other\computational_stats.json
Generating plots:  30%|███       | 3/10 [00:02<00:04,  1.74it/s]c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralbench\plots\normalized_summary.py:305: UserWarning: Skipping normalized-lines-summary plot: no tasks with baseline data
  warnings.warn(
Generating plots:  60%|██████    | 6/10 [00:02<00:00,  4.60it/s]c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralbench\plots\normalized_summary.py:305: UserWarning: Skipping normalized-lines-summary plot: no tasks with baseline data
  warnings.warn(
Generating plots: 100%|██████████| 10/10 [00:02<00:00,  4.63it/s]
INFO:neuralbench.plots.adaptation:plot_adaptation_comparison: no foundation-model rows -- skipping.



Outputs saved under D:\Foundation Challenge 2026\results\outputs in subfolders ('core', 'full', 'other', 'adaptation')


In [ ]:
# 2. Build the caches under CACHE_DIR (~13 GB): the preprocessed windows,
#    plus one frozen DINOv2-giant embedding per unique stimulus (~100 MB
#    for THINGS-EEG2, content-keyed and shared with the other image
#    tasks). The only --prepare of the four tracks that needs a GPU.
#    ~15 min for eegnet's cache, ~45 min for reve's, and ~10 min to embed
#    the 16740 stimuli, spread over 10 and 128 SLURM jobs respectively.
neuralbench eeg image --prepare

# 3. Sanity check before you queue anything: 2 epochs, a data subset, one
#    seed, always in-process, so progress lands in your terminal. ~2 min
#    on one V100 with the cache warm. Name the model you actually plan to
#    run -- a bare --debug takes the config default, which is EEGNet.
neuralbench eeg image -m eegnet --debug

# 4. Same check for the foundation model. REVE's weights are gated on the
#    HuggingFace Hub, so this needs an account and an accepted licence;
#    it is the cheapest place to discover that, because a queued run
#    reports the failure into a job log instead of your terminal.
neuralbench eeg image -m reve --debug

# 5. Full baseline -- task-specific model (EEGNet). ~2.5 h per seed, and
#    the default grid is three seeds (concurrent on SLURM).
neuralbench eeg image -m eegnet

# 6. Full baseline -- foundation model (REVE), fine-tuned end to end.
#    ~5.5 h per seed. ~69M parameters against EEGNet's ~1.5k, all of them
#    trainable here, so this one wants a datacentre GPU rather than a
#    laptop; it also preprocesses at 200 Hz against the 120 Hz default,
#    warming a second cache.
neuralbench eeg image -m reve

In [2]:
import neuralset as ns

study = ns.Study(
    name="Xu2025Alljoined",
    path=r"D:\Foundation Challenge 2026\data",
)

events = study.run()

images = events[events["type"] == "Image"].copy()

mask = images.duplicated(
    subset=["timeline", "start"],
    keep=False,
)

dups = images.loc[mask].sort_values(["timeline", "start"])

cols = [
    c for c in [
        "type",
        "timeline",
        "start",
        "duration",
        "filepath",
        "category",
        "subject",
        "session",
        "run",
    ]
    if c in dups.columns
]

print("Number of duplicated Image rows:", len(dups))
print("Available columns:", dups.columns.tolist())
print(dups[cols].to_string(index=False))

Extracting EDF parameters from D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\raw_eeg\sub-01\session_01\block_01\Subject 1, Session 1, Block 1 Recording_FLEX2_213075_2025.01.25T15.18.28.08.00.md.edf...
Setting channel info structure...
Creating raw.info structure...


c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:173: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = mne.channels.make_standard_montage("standard_1005")
c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:179: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage("standard_1005", on_missing="ignore", match_case=False)


Extracting EDF parameters from D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\raw_eeg\sub-01\session_01\block_01\Subject 1, Session 1, Block 1 Recording_FLEX2_213075_2025.01.25T15.18.28.08.00.md.edf...
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\raw_eeg\sub-01\session_01\block_02\Subject 1, Session 1, Block 2 Recording_FLEX2_213075_2025.01.25T15.24.00.08.00.md.edf...
Setting channel info structure...


c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:173: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = mne.channels.make_standard_montage("standard_1005")
c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:179: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage("standard_1005", on_missing="ignore", match_case=False)


Creating raw.info structure...
Extracting EDF parameters from D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\raw_eeg\sub-01\session_01\block_02\Subject 1, Session 1, Block 2 Recording_FLEX2_213075_2025.01.25T15.24.00.08.00.md.edf...
Setting channel info structure...


...

Extracting EDF parameters from D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\raw_eeg\sub-20\session_04\block_18\Subject 20, Session 4, Block 18 Recording_FLEX2_213075_2025.01.25T11.23.47.08.00.md.edf...
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\raw_eeg\sub-20\session_04\block_18\Subject 20, Session 4, Block 18 Recording_FLEX2_213075_2025.01.25T11.23.47.08.00.md.edf...
Setting channel info structure...
Creating raw.info structure...


c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:167: RuntimeWarning: Omitted 18 annotation(s) that were outside data range.
  raw = mne.io.read_raw(filepath)
c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:173: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = mne.channels.make_standard_montage("standard_1005")
c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:179: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage("standard_1005", on_missing="ignore", match_case=False)
c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:167: RuntimeWarning: Omitted 18 annotation(s) that were outside data range.
  raw = mne.io.read_

Extracting EDF parameters from D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\raw_eeg\sub-20\session_04\block_19\Subject 20, Session 4, Block 19 Recording_FLEX2_213075_2025.01.25T11.28.57.08.00.md.edf...
Setting channel info structure...
Creating raw.info structure...
Extracting EDF parameters from D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\raw_eeg\sub-20\session_04\block_19\Subject 20, Session 4, Block 19 Recording_FLEX2_213075_2025.01.25T11.28.57.08.00.md.edf...
Setting channel info structure...
Creating raw.info structure...


c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:173: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = mne.channels.make_standard_montage("standard_1005")
c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:179: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  raw.set_montage("standard_1005", on_missing="ignore", match_case=False)
c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:173: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  montage = mne.channels.make_standard_montage("standard_1005")
c:\Users\mcapo\anaconda3\envs\foundation2026\Lib\site-packages\neuralfetch\studies\xu2025alljoined.py:179: FutureWarning

Number of duplicated Image rows: 578
Available columns: ['type', 'start', 'duration', 'timeline', 'subject', 'session', 'task', 'run', 'filepath', 'frequency', 'description', 'label', 'value', 'deleted', 'orig_index', 'study', 'offset', 'caption', 'stop']
 type                                    timeline      start  duration                                                                            filepath            subject session run
Image  Xu2025Alljoined:run=1,session=1,subject=10 110.448626       0.1 D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\stimuli\images\16733.jpg Xu2025Alljoined/10       1   1
Image  Xu2025Alljoined:run=1,session=1,subject=10 110.448626       0.1 D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\stimuli\images\00001.jpg Xu2025Alljoined/10       1   1
Image   Xu2025Alljoined:run=1,session=3,subject=4  41.155483       0.1 D:\Foundation Challenge 2026\data\Xu2025Alljoined\download\stimuli\images\16704.jpg  Xu2025Alljoined/4       3   1
